In [1]:
#Install + Imports
!pip install transformers accelerate -q

import pandas as pd, numpy as np, torch, json, os, shutil
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:

# Configuration
CONFIG = {
    'model_name': 'distilbert-base-uncased',
    'max_len':    128,
    'batch_size': 64,     # Kaggle GPU T4 ke liye
    'epochs':     3,
    'lr':         2e-5,
    'threshold':  0.5,
    'save_path':  '/kaggle/working/saved_model/',
}
os.makedirs(CONFIG['save_path'], exist_ok=True)

In [3]:

# Load Data

df = pd.read_csv('/kaggle/input/datasets/harsh0812/dataset/train.csv')
label_cols = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']
df['label'] = df[label_cols].max(axis=1).astype(int)

train_df, val_df = train_test_split(
    df[['comment_text','label']], test_size=0.15,
    stratify=df['label'], random_state=42
)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")

Train: 135635 | Val: 23936


In [4]:
# Dataset Class

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

class JigsawDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(str(self.texts[idx]), max_length=CONFIG['max_len'],
                        padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds = JigsawDataset(train_df['comment_text'], train_df['label'])
val_ds   = JigsawDataset(val_df['comment_text'],   val_df['label'])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:

# Train
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['model_name'], num_labels=2)
model.to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(logits, axis=1)
    return {
        'roc_auc':  round(roc_auc_score(labels, probs[:,1]), 4),
        'accuracy': round(accuracy_score(labels, preds), 4)
    }

args = TrainingArguments(
    output_dir=CONFIG['save_path']+'checkpoints',
    num_train_epochs=CONFIG['epochs'],
    per_device_train_batch_size=CONFIG['batch_size'],
    per_device_eval_batch_size=CONFIG['batch_size']*2,
    learning_rate=CONFIG['lr'],
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',       
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='roc_auc',
    fp16=True,
    logging_steps=200,
    report_to='none',

)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()
results = trainer.evaluate()
print(f"\nROC-AUC: {results['eval_roc_auc']} | Accuracy: {results['eval_accuracy']}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were

Epoch,Training Loss,Validation Loss,Roc Auc,Accuracy
1,0.162824,0.200514,0.984300,0.966200
2,0.137028,0.200326,0.985700,0.963000
3,0.095668,0.199973,0.984800,0.966300


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



ROC-AUC: 0.9857 | Accuracy: 0.9628


In [6]:
# Save + ZIP 

trainer.save_model(CONFIG['save_path'] + 'model/')
tokenizer.save_pretrained(CONFIG['save_path'] + 'tokenizer/')

json.dump({
    'model_name':   CONFIG['model_name'],
    'max_len':      CONFIG['max_len'],
    'threshold':    CONFIG['threshold'],
    'val_roc_auc':  results['eval_roc_auc'],
    'val_accuracy': results['eval_accuracy'],
}, open(CONFIG['save_path'] + 'config.json', 'w'), indent=2)

# ZIP banao — ek click mein download hoga
shutil.make_archive('/kaggle/working/saved_model', 'zip', '/kaggle/working/saved_model/')
print("✅ /kaggle/working/saved_model.zip — Download this!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ /kaggle/working/saved_model.zip — Download this!
